In [2]:
%pip install duckdb pandas

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/13.1 MB ? eta -:--:--
   --- ------------------------------------ 1.0/13.1 MB 5.7 MB/s eta 0:00:03
   ------ --------------------------------- 2.1/13.1 MB 5.8 MB/s eta 0:00:02
   -------- ------------------------------- 2.9/13.1 MB 4.8 MB/s eta 0:00:03
   -------------- ------------------------- 4.7/13.1 MB 5.9 MB/s eta 0:00:02
   ------------------- -------------------- 6.3/13.1 MB 6.2 MB/s eta 0:00:02
   ------------------------ --------------- 8.1/13.1 MB 6.8 MB/s eta 0:00:01
   ------------------------ --------------- 8.1/13.1 MB 6.8 MB/s eta 0:00:01
   -------------------------- ------------- 8.7/13.1 MB 5.4 MB/s eta 0:00:01
   ------------------------------- -------- 10.2/13.1 MB 5.6 MB/s eta 0:00:01
   ---------------------------------- ----- 11.3/13.1 MB 5.5 MB/s eta 0:00:01
   -------------------------------------- - 12.6/13.1 MB 5.6 MB/s eta 0:00:01
  

In [1]:
import duckdb
import pandas as pd

In [4]:
base_table_csv = "base_table.csv"
output_csv = "delivery_performance.csv"

# Wraping the existing query inside an EXPORT statement
optimized_query = f"""
COPY (
    SELECT 
        order_id,
        order_purchase_timestamp,
        shipping_limit_date,
        order_delivered_customer_date,
        EPOCH(order_delivered_customer_date - order_purchase_timestamp) / 86400.0 AS total_delivery_time_days,
        price,
        CASE 
            WHEN order_delivered_customer_date IS NULL THEN 'Not Delivered Yet'
            WHEN order_delivered_customer_date > shipping_limit_date THEN 'Late'
            WHEN EPOCH(order_delivered_customer_date - order_purchase_timestamp) / 86400.0 <= 3.0 THEN 'Quick'
            ELSE 'In Time'
        END AS delivery_status,
        geolocation_lat,
        geolocation_lng,
        customer_city,
        customer_state,
        customer_zip_code_prefix
    FROM read_csv_auto('{base_table_csv}')
) TO '{output_csv}' (HEADER, DELIMITER ',');
"""

print(f"DuckDB is processing and streaming directly to {output_csv}...")
duckdb.query(optimized_query)
print("Done! Completely processed without memory bloat.")

DuckDB is processing and streaming directly to delivery_performance.csv...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Done! Completely processed without memory bloat.


In [6]:
# Sample 1,000 rows from the large output CSV for quick analysis
large_results_csv = "delivery_performance.csv"
sample_output_csv = "sample_delivery_performance_1k.csv"

sample_query = f"""
COPY (
    SELECT * FROM read_csv_auto('{large_results_csv}') 
    USING SAMPLE 1000 ROWS
) TO '{sample_output_csv}' (HEADER, DELIMITER ',');
"""

print(f"Sampling 1,000 rows directly from {large_results_csv}...")
duckdb.query(sample_query)
print(f"Done! Created {sample_output_csv}")

Sampling 1,000 rows directly from delivery_performance.csv...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Done! Created sample_delivery_performance_1k.csv
